In [ ]:
!pip install uvicorn starlette faster-whisper deep-translator pyngrok websockets -q

In [ ]:
import threading, numpy as np, json, time, queue
from starlette.applications import Starlette
from starlette.routing import Route, WebSocketRoute
from starlette.responses import PlainTextResponse
from starlette.websockets import WebSocket, WebSocketDisconnect
import uvicorn
from faster_whisper import WhisperModel
from deep_translator import GoogleTranslator
from pyngrok import conf, ngrok
from google.colab import userdata
import asyncio

conf.get_default().auth_token = userdata.get('NGROK_TOKEN')
model = WhisperModel("medium", device="cuda", compute_type="float16")

SAMPLE_RATE      = 16000
FRAMES_PER_CHUNK = int(SAMPLE_RATE * 1.5)   # 1s chunks — more context = less hallucination
MIN_ENERGY       = 0.018                     # noise gate — raise to 0.025 if still noisy

LANGUAGE_NAMES = {
    "en":"English","hi":"Hindi","ta":"Tamil","te":"Telugu",
    "bn":"Bengali","mr":"Marathi","gu":"Gujarati","kn":"Kannada",
    "ml":"Malayalam","pa":"Punjabi","ur":"Urdu","or":"Odia",
    "as":"Assamese","ne":"Nepali","si":"Sinhala",
    # European languages
    "fr":"French","de":"German","es":"Spanish","it":"Italian",
    "pt":"Portuguese","nl":"Dutch","pl":"Polish","ru":"Russian",
    "uk":"Ukrainian","sv":"Swedish","no":"Norwegian","da":"Danish",
    "fi":"Finnish","cs":"Czech","ro":"Romanian","hu":"Hungarian",
    # Asian languages
    "ja":"Japanese","ko":"Korean","th":"Thai",
    "vi":"Vietnamese","id":"Indonesian","ms":"Malay",
    # Middle Eastern
    "ar":"Arabic","fa":"Persian","tr":"Turkish","he":"Hebrew",
}

rooms: dict[str, dict] = {}
rooms_lock = threading.Lock()

# ── Single transcription queue — one GPU job at a time ────────────────────
transcription_queue = queue.Queue(maxsize=3)  # maxsize=3 prevents backlog buildup

def rms_energy(audio: np.ndarray) -> float:
    return float(np.sqrt(np.mean(audio ** 2)))

def is_garbage(text: str) -> bool:
    cleaned = text.strip().lower().strip(".,!?-– ")
    if not cleaned:
        return True
    # Too short to be real speech
    if len(cleaned) < 3:
        return True
    # Repeated characters like "aaaa" or "hhhh"
    if len(set(cleaned.replace(" ", ""))) <= 2:
        return True
    # Only punctuation or symbols
    if all(not c.isalnum() for c in cleaned):
        return True
    return False

def transcription_worker():
    print("[Worker] Transcription worker ready")
    while True:
        item = transcription_queue.get()
        if item is None:
            break

        audio_chunk, spoken, room_id, loop, last_ref, callback = item

        try:
            segments, info = model.transcribe(
                audio_chunk,
                language=spoken,
                beam_size=1,
                best_of=1,
                vad_filter=True,
                without_timestamps=True,
                vad_parameters=dict(
                    threshold=0.65,               # was 0.3 — much stricter speech detection
                    min_speech_duration_ms=300,   # ignore sounds shorter than 300ms
                    min_silence_duration_ms=600,  # wait longer before cutting a segment
                    speech_pad_ms=200,
                ),
                condition_on_previous_text=False,
                no_speech_threshold=0.65,         # drop low-confidence segments
                log_prob_threshold=-0.7,          # drop uncertain output
                compression_ratio_threshold=1.8,  # drop repetitive/looping output
                temperature=0.0,                  # deterministic — no random sampling noise
            )

            # Only keep segments with decent confidence
            good_segments = []
            for seg in segments:
                # Skip segments where whisper itself is uncertain
                if seg.no_speech_prob > 0.5:
                    continue
                if seg.avg_logprob < -0.7:
                    continue
                t = seg.text.strip()
                if t:
                    good_segments.append(t)

            text = " ".join(good_segments)

            if is_garbage(text):
                continue
            if text == last_ref[0]:
                continue

            last_ref[0] = text
            print(f"[{LANGUAGE_NAMES.get(spoken,spoken)} → room '{room_id}'] {text}")

            asyncio.run_coroutine_threadsafe(
                callback(room_id, text, spoken), loop
            )

        except Exception as e:
            print(f"[Worker] Error: {e}")
        finally:
            transcription_queue.task_done()

worker_thread = threading.Thread(target=transcription_worker, daemon=True)
worker_thread.start()

async def broadcast_to_room(room_id: str, text: str, spoken: str):
    with rooms_lock:
        clients = dict(rooms.get(room_id, {}))

    tasks = []
    for ws, cfg in clients.items():
        target   = cfg.get("target_lang", "en")
        tgt_name = LANGUAGE_NAMES.get(target, target)
        translated = text
        if spoken != target:
            try:
                translated = GoogleTranslator(source=spoken, target=target).translate(text)
            except Exception as e:
                print(f"Translation error: {e}")
        tasks.append(ws.send_text(json.dumps({
            "type":     "subtitle",
            "text":     translated,
            "language": tgt_name,
        })))

    if tasks:
        results = await asyncio.gather(*tasks, return_exceptions=True)
        for r in results:
            if isinstance(r, Exception):
                print(f"Broadcast error: {r}")

async def health(request):
    return PlainTextResponse("OK")

async def subtitle_ws(websocket: WebSocket):
    await websocket.accept()

    config   = {"spoken_lang": "hi", "target_lang": "en"}
    buf      = np.array([], dtype=np.float32)
    last_ref = [""]
    room_id  = "default"
    loop     = asyncio.get_event_loop()

    with rooms_lock:
        rooms.setdefault(room_id, {})[websocket] = config
    print(f"[+] Client connected → room '{room_id}'")

    try:
        while True:
            message = await websocket.receive()

            if message["type"] == "websocket.disconnect":
                break

            elif message["type"] == "websocket.receive":
                if message.get("text"):
                    try:
                        data = json.loads(message["text"])
                        if data.get("type") == "config":
                            new_room = data.get("room_id", "default")
                            config = {
                                "spoken_lang": data.get("spoken_lang", "hi"),
                                "target_lang":  data.get("target_lang",  "en"),
                            }
                            with rooms_lock:
                                if room_id in rooms and websocket in rooms[room_id]:
                                    del rooms[room_id][websocket]
                                    if not rooms[room_id]:
                                        del rooms[room_id]
                                room_id = new_room
                                rooms.setdefault(room_id, {})[websocket] = config
                            print(f"[Config] room={room_id} spoken={config['spoken_lang']} target={config['target_lang']} | size={len(rooms.get(room_id, {}))}")
                    except Exception as e:
                        print(f"Bad config: {e}")

                elif message.get("bytes"):
                    chunk = np.frombuffer(message["bytes"], dtype=np.float32)
                    buf = np.concatenate([buf, chunk])

                    if len(buf) >= FRAMES_PER_CHUNK:
                        spoken     = config.get("spoken_lang", "hi")
                        data_chunk = buf[:FRAMES_PER_CHUNK]
                        buf        = buf[FRAMES_PER_CHUNK // 2:]

                        # Noise gate — drop silent audio before it hits the GPU
                        if rms_energy(data_chunk) < MIN_ENERGY:
                            continue

                        # If queue is full drop oldest chunk — prefer fresh audio
                        if transcription_queue.full():
                            try:
                                transcription_queue.get_nowait()
                            except queue.Empty:
                                pass

                        transcription_queue.put((
                            data_chunk,
                            spoken,
                            room_id,
                            loop,
                            last_ref,
                            broadcast_to_room,
                        ))

    except WebSocketDisconnect:
        pass
    finally:
        with rooms_lock:
            if room_id in rooms and websocket in rooms[room_id]:
                del rooms[room_id][websocket]
                if not rooms[room_id]:
                    del rooms[room_id]
        print(f"[-] Client disconnected from room '{room_id}'")

app = Starlette(routes=[
    Route('/', health),
    WebSocketRoute('/subtitles', subtitle_ws),
])

cfg    = uvicorn.Config(app, host="0.0.0.0", port=8765, log_level="warning")
server = uvicorn.Server(cfg)

thread = threading.Thread(target=server.run, daemon=True)
thread.start()
time.sleep(3)

import requests
r = requests.get("http://localhost:8765/")
print("Local server test:", r.status_code, r.text)

tunnel = ngrok.connect(
    addr=8765,
    proto="http",
    bind_tls=True,
    inspect=False,
    domain="buffy-nonrestricted-pablo.ngrok-free.dev",
)
print(f"\n{'='*50}")
print(f"{'='*50}")
print("Ready. Waiting for connections...")

while True:
    time.sleep(55)
    print(".", end="", flush=True)

In [ ]:
import os, time
os.system("fuser -k 8765/tcp")
time.sleep(2)
print("Port cleared")